# NBA One-Game Lineup Stint Pipeline (End-to-End)

This notebook reproduces the full pipeline for one game using player IDs as the source of truth.

1. Load cached `playbyplayv3` actions
2. Build grouped timestamps and grouped same-clock substitutions
3. Infer period-opening lineups (Q1 from live starters, Q2/Q3/Q4 inferred)
4. Validate inferred lineups
5. Build final `fact_lineup_stint`
6. Run quality checks

In [ ]:
import re
import pandas as pd

from src.ingestion.pbp_json import load_pbp_actions_dataframe
from src.processing.clock import parse_clock_seconds_remaining
from nba_api.live.nba.endpoints import boxscore as live_boxscore

pd.set_option("display.max_colwidth", 120)

In [ ]:
# Config (same game used in prior work)
GAME_ID = "0042500111"

# Load and sort actions
pbp_df = load_pbp_actions_dataframe(GAME_ID).copy()
pbp_df = pbp_df.sort_values("actionNumber", kind="mergesort").reset_index(drop=True)
pbp_df["clock_seconds_remaining"] = pbp_df["clock"].apply(parse_clock_seconds_remaining)

print("Loaded actions:", len(pbp_df))
display(pbp_df[["actionNumber", "period", "clock", "teamId", "personId", "playerName", "actionType", "description", "scoreHome", "scoreAway"]].head(10))

In [ ]:
# Player master from live boxscore (for IDs + readable names + Q1 starters)
live_bs = live_boxscore.BoxScore(game_id=GAME_ID).get_dict()["game"]
home_team_id = int(live_bs["homeTeam"]["teamId"])
away_team_id = int(live_bs["awayTeam"]["teamId"])

player_rows = []
for team_key, tid in [("homeTeam", home_team_id), ("awayTeam", away_team_id)]:
    for p in live_bs[team_key]["players"]:
        player_rows.append({
            "team_id": int(tid),
            "player_id": int(p["personId"]),
            "display_name": str(p.get("name") or p.get("familyName") or p.get("personId")),
            "starter": int(p.get("starter", 0)) == 1,
        })

player_master_df = pd.DataFrame(player_rows).drop_duplicates(subset=["team_id", "player_id"]).reset_index(drop=True)
player_id_to_name = dict(zip(player_master_df["player_id"], player_master_df["display_name"]))

def canon_name(s):
    return re.sub(r"[^a-z0-9]", "", str(s).lower())

normalized_name_to_player_id = {
    (int(r["team_id"]), canon_name(r["display_name"])): int(r["player_id"])
    for _, r in player_master_df.iterrows()
}

q1_home_starters = set(player_master_df[(player_master_df["team_id"] == home_team_id) & (player_master_df["starter"])]["player_id"].astype(int).tolist())
q1_away_starters = set(player_master_df[(player_master_df["team_id"] == away_team_id) & (player_master_df["starter"])]["player_id"].astype(int).tolist())

assert len(q1_home_starters) == 5 and len(q1_away_starters) == 5
print("Q1 HOME starters:", sorted(player_id_to_name.get(i, i) for i in q1_home_starters))
print("Q1 AWAY starters:", sorted(player_id_to_name.get(i, i) for i in q1_away_starters))

In [ ]:
# Group same-clock substitution rows
sub_df = pbp_df[pbp_df["actionType"].eq("Substitution")].copy()
SUB_PATTERN = re.compile(r"^SUB:\\s*(?P<player_in>.+?)\\s+FOR\\s+(?P<player_out>.+?)\\s*$", re.IGNORECASE)
parsed = sub_df["description"].astype(str).str.extract(SUB_PATTERN)

sub_df["team_id"] = pd.to_numeric(sub_df["teamId"], errors="coerce").astype("Int64")
sub_df["player_out_id"] = pd.to_numeric(sub_df["personId"], errors="coerce").astype("Int64")
sub_df["player_in_name"] = parsed["player_in"].astype(str).str.strip()
sub_df["player_in_id"] = sub_df.apply(
    lambda r: normalized_name_to_player_id.get((int(r["team_id"]), canon_name(r["player_in_name"]))) if pd.notna(r["team_id"]) else None,
    axis=1,
)

sub_events_clean = sub_df[["actionNumber", "period", "clock", "team_id", "player_out_id", "player_in_id", "description"]].copy()
sub_events_clean["player_out_id"] = pd.to_numeric(sub_events_clean["player_out_id"], errors="coerce").astype("Int64")
sub_events_clean["player_in_id"] = pd.to_numeric(sub_events_clean["player_in_id"], errors="coerce").astype("Int64")

sub_groups_df = (
    sub_events_clean.groupby(["period", "clock"], sort=False)
    .apply(lambda g: pd.Series({"n_subs": len(g), "substitutions": g.sort_values("actionNumber").to_dict("records")}))
    .reset_index()
)

sub_group_lookup = {(int(r["period"]), r["clock"]): r["substitutions"] for _, r in sub_groups_df.iterrows()}

# Group all events by timestamp
event_groups_df = (
    pbp_df.groupby(["period", "clock", "clock_seconds_remaining"], sort=False)
    .agg(n_events=("actionNumber", "count"), action_numbers=("actionNumber", list), action_types=("actionType", list))
    .reset_index()
    .sort_values(["period", "clock_seconds_remaining"], ascending=[True, False])
    .reset_index(drop=True)
)

sub_keys = set(zip(sub_groups_df["period"].astype(int), sub_groups_df["clock"]))
event_groups_df["has_substitution"] = event_groups_df.apply(lambda r: (int(r["period"]), r["clock"]) in sub_keys, axis=1)

print("Sub batches:", len(sub_groups_df), "| Event groups:", len(event_groups_df))
display(sub_groups_df.head(8))

In [ ]:
# Helpers for lineup inference and validation
def _to_int(x):
    try:
        return int(x)
    except Exception:
        return None

def ids_to_names(ids):
    return [player_id_to_name.get(int(i), f"id:{int(i)}") for i in sorted(set(int(x) for x in ids))]

def get_first_sub_roles_for_period(period):
    q = sub_groups_df[sub_groups_df["period"].eq(period)].copy()
    if q.empty:
        return {"home": {}, "away": {}, "home_out": set(), "home_in": set(), "away_out": set(), "away_in": set()}
    q["sec"] = q["clock"].apply(parse_clock_seconds_remaining)
    q = q.sort_values("sec", ascending=False)

    home_roles, away_roles = {}, {}
    for _, row in q.iterrows():
        for sub in row["substitutions"]:
            team_id = _to_int(sub.get("team_id"))
            out_id = _to_int(sub.get("player_out_id"))
            in_id = _to_int(sub.get("player_in_id"))
            roles = home_roles if team_id == home_team_id else away_roles if team_id == away_team_id else None
            if roles is None:
                continue
            if out_id is not None and out_id not in roles:
                roles[out_id] = "OUT"
            if in_id is not None and in_id not in roles:
                roles[in_id] = "IN"

    return {
        "home": home_roles,
        "away": away_roles,
        "home_out": {pid for pid, role in home_roles.items() if role == "OUT"},
        "home_in": {pid for pid, role in home_roles.items() if role == "IN"},
        "away_out": {pid for pid, role in away_roles.items() if role == "OUT"},
        "away_in": {pid for pid, role in away_roles.items() if role == "IN"},
    }

def get_first_sub_clock(period):
    q = sub_groups_df[sub_groups_df["period"].eq(period)].copy()
    if q.empty:
        return None
    q["sec"] = q["clock"].apply(parse_clock_seconds_remaining)
    return q.sort_values("sec", ascending=False).iloc[0]["clock"]

def get_players_seen_before_first_sub(period, first_sub_clock):
    q = pbp_df[pbp_df["period"].eq(period)].copy()
    if "clock_seconds_remaining" not in q.columns:
        q["clock_seconds_remaining"] = q["clock"].apply(parse_clock_seconds_remaining)
    fs = parse_clock_seconds_remaining(first_sub_clock) if first_sub_clock else None
    if fs is not None:
        q = q[(q["clock_seconds_remaining"] < 720.0) & (q["clock_seconds_remaining"] > fs)]
    q = q[
        (~q["actionType"].astype(str).str.lower().eq("period"))
        & (~q["actionType"].astype(str).str.lower().eq("substitution"))
    ].copy()
    q["personId_int"] = pd.to_numeric(q["personId"], errors="coerce")
    q["teamId_int"] = pd.to_numeric(q["teamId"], errors="coerce")
    q = q[(q["personId_int"].notna()) & (q["personId_int"] != 0) & (q["teamId_int"].isin([home_team_id, away_team_id]))]
    home_seen = set(q[q["teamId_int"].eq(home_team_id)]["personId_int"].astype(int).tolist())
    away_seen = set(q[q["teamId_int"].eq(away_team_id)]["personId_int"].astype(int).tolist())
    return home_seen, away_seen

def apply_sub_batch_ids(home_ids, away_ids, sub_batch):
    home, away = set(home_ids), set(away_ids)
    debug = []
    for sub in sub_batch or []:
        team_id = _to_int(sub.get("team_id"))
        out_id = _to_int(sub.get("player_out_id"))
        in_id = _to_int(sub.get("player_in_id"))
        lineup = home if team_id == home_team_id else away if team_id == away_team_id else None
        if lineup is None:
            continue
        ok = (out_id in lineup) and (in_id not in lineup) and (in_id is not None)
        if ok:
            lineup.remove(out_id)
            lineup.add(in_id)
        debug.append({"team_id": team_id, "out_id": out_id, "in_id": in_id, "applied": ok})
    return home, away, debug

In [ ]:
# Infer period_start_lineups_inferred (Q1 official + Q2/Q3/Q4 inferred)
period_start_lineups_inferred = {1: {"home": set(q1_home_starters), "away": set(q1_away_starters)}}

def period_end_lineups_from_openers(period, home_open, away_open):
    q = sub_groups_df[sub_groups_df["period"].eq(period)].copy()
    if q.empty:
        return set(home_open), set(away_open)
    q["sec"] = q["clock"].apply(parse_clock_seconds_remaining)
    q = q.sort_values("sec", ascending=False)
    h, a = set(home_open), set(away_open)
    for _, row in q.iterrows():
        h, a, _ = apply_sub_batch_ids(h, a, sub_group_lookup.get((int(row["period"]), row["clock"]), []))
    return h, a

prev_home_end, prev_away_end = period_end_lineups_from_openers(1, q1_home_starters, q1_away_starters)
debug_rows = []

for period in [2, 3, 4]:
    roles = get_first_sub_roles_for_period(period)
    first_sub_clock = get_first_sub_clock(period)
    seen_home, seen_away = get_players_seen_before_first_sub(period, first_sub_clock)

    fallback_home = set(q1_home_starters if period == 3 else prev_home_end)
    fallback_away = set(q1_away_starters if period == 3 else prev_away_end)

    home_candidates = list(sorted(roles["home_out"])) + list(sorted(seen_home - roles["home_in"])) + list(sorted(fallback_home - roles["home_in"]))
    away_candidates = list(sorted(roles["away_out"])) + list(sorted(seen_away - roles["away_in"])) + list(sorted(fallback_away - roles["away_in"]))

    inferred_home, inferred_away = [], []
    for pid in home_candidates:
        if pid not in inferred_home:
            inferred_home.append(pid)
        if len(inferred_home) == 5:
            break
    for pid in away_candidates:
        if pid not in inferred_away:
            inferred_away.append(pid)
        if len(inferred_away) == 5:
            break

    period_start_lineups_inferred[period] = {"home": set(inferred_home), "away": set(inferred_away)}
    debug_rows.append({
        "period": period,
        "first_sub_clock": first_sub_clock,
        "home_out": ids_to_names(roles["home_out"]),
        "home_in": ids_to_names(roles["home_in"]),
        "home_seen": ids_to_names(seen_home),
        "home_final": ids_to_names(inferred_home),
        "away_out": ids_to_names(roles["away_out"]),
        "away_in": ids_to_names(roles["away_in"]),
        "away_seen": ids_to_names(seen_away),
        "away_final": ids_to_names(inferred_away),
    })

    prev_home_end, prev_away_end = period_end_lineups_from_openers(period, inferred_home, inferred_away)

display(pd.DataFrame(debug_rows))

In [ ]:
# Strong validation: first 3 sub batches in each period must apply cleanly
def validate_first_n_sub_batches(period, n_batches=3):
    q = sub_groups_df[sub_groups_df["period"].eq(period)].copy()
    q["sec"] = q["clock"].apply(parse_clock_seconds_remaining)
    q = q.sort_values("sec", ascending=False).head(n_batches)
    h = set(period_start_lineups_inferred[period]["home"])
    a = set(period_start_lineups_inferred[period]["away"])
    rows = []
    for _, row in q.iterrows():
        for sub in sub_group_lookup.get((int(row["period"]), row["clock"]), []):
            team_id = _to_int(sub.get("team_id"))
            out_id = _to_int(sub.get("player_out_id"))
            in_id = _to_int(sub.get("player_in_id"))
            lineup = h if team_id == home_team_id else a if team_id == away_team_id else None
            if lineup is None:
                continue
            out_found = out_id in lineup
            in_already = in_id in lineup if in_id is not None else True
            ok = out_found and (not in_already) and (in_id is not None)
            if ok:
                lineup.remove(out_id)
                lineup.add(in_id)
            rows.append({
                "period": period,
                "clock": row["clock"],
                "outgoing_found": out_found,
                "incoming_already_present": in_already,
                "size_ok": len(h) == 5 and len(a) == 5,
                "applied": ok,
            })
    out = pd.DataFrame(rows)
    out["pass"] = out["outgoing_found"] & (~out["incoming_already_present"]) & out["size_ok"]
    return out

validation_df = pd.concat([validate_first_n_sub_batches(p, 3) for p in [2, 3, 4]], ignore_index=True)
print("Validation pass rows:", int(validation_df["pass"].sum()), "/", len(validation_df))
display(validation_df)

In [ ]:
# Build final fact_lineup_stint
def score_snapshot(period, clock, last_known=(0, 0)):
    rows = pbp_df[(pbp_df["period"].eq(period)) & (pbp_df["clock"].eq(clock))].copy()
    if rows.empty:
        return last_known
    sh = pd.to_numeric(rows["scoreHome"], errors="coerce").dropna()
    sa = pd.to_numeric(rows["scoreAway"], errors="coerce").dropna()
    return (int(sh.iloc[-1]) if len(sh) else int(last_known[0]), int(sa.iloc[-1]) if len(sa) else int(last_known[1]))

def build_row(period, start_clock, end_clock, home_ids, away_ids, sh0, sa0, sh1, sa1, ended_by_sub):
    s0 = float(parse_clock_seconds_remaining(start_clock))
    s1 = float(parse_clock_seconds_remaining(end_clock))
    home_ids = sorted(int(i) for i in home_ids)
    away_ids = sorted(int(i) for i in away_ids)
    return {
        "game_id": GAME_ID,
        "period": int(period),
        "start_clock": start_clock,
        "end_clock": end_clock,
        "start_seconds_remaining": s0,
        "end_seconds_remaining": s1,
        "duration_seconds": s0 - s1,
        "home_team_id": home_team_id,
        "away_team_id": away_team_id,
        "home_lineup_ids": home_ids,
        "away_lineup_ids": away_ids,
        "home_lineup_names": [player_id_to_name.get(i, f"id:{i}") for i in home_ids],
        "away_lineup_names": [player_id_to_name.get(i, f"id:{i}") for i in away_ids],
        "score_home_start": int(sh0),
        "score_away_start": int(sa0),
        "score_home_end": int(sh1),
        "score_away_end": int(sa1),
        "ended_by_substitution": bool(ended_by_sub),
    }

rows = []
last_known_score = (0, 0)
for period in [1, 2, 3, 4]:
    home_state = set(period_start_lineups_inferred[period]["home"])
    away_state = set(period_start_lineups_inferred[period]["away"])
    start_clock = "PT12M00.00S"
    sh_start, sa_start = score_snapshot(period, start_clock, last_known_score)
    cur_start_clock = start_clock
    cur_sh, cur_sa = sh_start, sa_start

    period_groups = event_groups_df[event_groups_df["period"].eq(period)].sort_values("clock_seconds_remaining", ascending=False)
    for _, g in period_groups.iterrows():
        clock = g["clock"]
        sh_here, sa_here = score_snapshot(period, clock, last_known_score)
        last_known_score = (sh_here, sa_here)
        if not bool(g["has_substitution"]):
            continue
        rows.append(build_row(period, cur_start_clock, clock, home_state, away_state, cur_sh, cur_sa, sh_here, sa_here, True))
        home_state, away_state, _ = apply_sub_batch_ids(home_state, away_state, sub_group_lookup.get((period, clock), []))
        cur_start_clock = clock
        cur_sh, cur_sa = sh_here, sa_here

    end_clock = "PT00M00.00S"
    sh_end, sa_end = score_snapshot(period, end_clock, last_known_score)
    last_known_score = (sh_end, sa_end)
    rows.append(build_row(period, cur_start_clock, end_clock, home_state, away_state, cur_sh, cur_sa, sh_end, sa_end, False))

fact_lineup_stint = pd.DataFrame(rows)
print("fact_lineup_stint rows:", len(fact_lineup_stint))
display(fact_lineup_stint.head(20))

In [ ]:
# Quality checks
neg_dur = int((fact_lineup_stint["duration_seconds"] < 0).sum())
zero_dur = int((fact_lineup_stint["duration_seconds"] == 0).sum())
bad_home = int(fact_lineup_stint["home_lineup_ids"].apply(lambda x: len(set(x)) != 5).sum())
bad_away = int(fact_lineup_stint["away_lineup_ids"].apply(lambda x: len(set(x)) != 5).sum())
total_sec = float(fact_lineup_stint["duration_seconds"].sum())

# Lineup changes should only happen at substitution boundaries
f = fact_lineup_stint.sort_values(["period", "start_seconds_remaining"], ascending=[True, False]).reset_index(drop=True)
boundary_issues = 0
for i in range(len(f) - 1):
    a, b = f.iloc[i], f.iloc[i + 1]
    if int(a["period"]) != int(b["period"]):
        continue
    lineup_changed = (set(a["home_lineup_ids"]) != set(b["home_lineup_ids"])) or (set(a["away_lineup_ids"]) != set(b["away_lineup_ids"]))
    if lineup_changed and (not bool(a["ended_by_substitution"]) or str(a["end_clock"]) != str(b["start_clock"])):
        boundary_issues += 1

print("Negative durations:", neg_dur)
print("Zero durations:", zero_dur)
print("Bad home lineup size:", bad_home)
print("Bad away lineup size:", bad_away)
print("Total regulation seconds:", total_sec, "(target=2880)")
print("Boundary sanity issues:", boundary_issues)